In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for supplier invoice fact table ETL logic in Databricks Unity Catalog
# Purpose: Convert Spark SQL logic for fact_wf_supplier_invoice_orafin to PySpark DataFrame transformations
# Author: Giang Nguyen
# Date: 2025-10-27
# Description: This script reads, transforms, and validates supplier invoice data from multiple Unity Catalog tables, applies window functions, joins, filters, and business logic, and writes the result to the target table in the specified format and partitioning. It includes error handling, schema enforcement, and data quality checks.

# from pyspark.sql import SparkSession  # SparkSession is already available in Databricks
from pyspark.sql import functions as F  
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, FloatType, ShortType, LongType, DateType, TimestampType
)  # pip install pyspark
from pyspark.sql.window import Window  
from datetime import datetime  

# -- Setup: Read input parameters from dbutils.widgets
target_table_path = dbutils.widgets.get("target_table_path")
partition_key = dbutils.widgets.get("partition")
table_format = dbutils.widgets.get("table_format")
compression = dbutils.widgets.get("compression")
table_name = dbutils.widgets.get("table_name")
unity_catalog = dbutils.widgets.get("unity_catalog")
environment = dbutils.widgets.get("environment")
project = dbutils.widgets.get("project")
load_type = dbutils.widgets.get("load_type")
view_unity_catalog_name = dbutils.widgets.get("view_unity_catalog_name")
raw_unity_catalog = dbutils.widgets.get("raw_unity_catalog")
raw_unity_catalog_hist = dbutils.widgets.get("raw_unity_catalog_hist")
config_unity_catalog = dbutils.widgets.get("config_unity_catalog")
edp_lkp_unity_catalog = dbutils.widgets.get("edp_lkp_unity_catalog")
dims_unity_catalog = dbutils.widgets.get("dims_unity_catalog")

# -- Setup: Set Spark configs for dynamic partition overwrite
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("raw_catalog.schema", raw_unity_catalog)
spark.conf.set("unity_catalog.schema", unity_catalog)

# -- Setup: Get jobId and jobUrl for logging
try:
    jobId = dbutils.notebook.entry_point.getDbutils().notebook().getContext().jobId().get()
except Exception:
    jobId = -123
# get_basejob_url and read_control_table are assumed imported from DA_CCG_RSD_EU/ReusableFunctions
baseUrl, JobsAPI_Secret = get_basejob_url(environment)
jobUrl = baseUrl + "/#job/" + str(jobId) + "/run/1"
start_time = datetime.now()

# -- Setup: Read control table entry
control_table = f"{config_unity_catalog}.{cl_control_table}"
ctrl_tbl_entry = read_control_table(project, table_name, load_type, control_table)

def validate_schema(df, expected_schema):
    """
    Validates that the DataFrame schema matches the expected schema (excluding nullable).
    Args:
        df (DataFrame): DataFrame to validate.
        expected_schema (StructType): Expected schema.
    Returns:
        bool: True if schema matches, False otherwise.
    """
    df_fields = [(f.name, f.dataType) for f in df.schema.fields]
    exp_fields = [(f.name, f.dataType) for f in expected_schema.fields]
    return df_fields == exp_fields

def enforce_schema(df, expected_schema):
    """
    Enforces column order and types on the DataFrame.
    Args:
        df (DataFrame): Input DataFrame.
        expected_schema (StructType): Target schema.
    Returns:
        DataFrame: DataFrame with columns in correct order and types.
    """
    for field in expected_schema.fields:
        df = df.withColumn(field.name, F.col(field.name).cast(field.dataType))
    df = df.select([field.name for field in expected_schema.fields])
    return df

# -- Define expected schema for fact_wf_supplier_invoice_orafin
expected_schema = StructType([
    StructField("document_type", StringType(), True),
    StructField("txn_ref_nbr", StringType(), True),
    StructField("invc_entry_period", StringType(), True),
    StructField("po_nbr", StringType(), True),
    StructField("po_line_nbr", StringType(), True),
    StructField("src_sys_cd", StringType(), True),
    StructField("vchr_nbr", StringType(), True),
    StructField("vchr_line_nbr", StringType(), True),
    StructField("fscl_yr_nbr", StringType(), True),
    StructField("vchr_type_cd", StringType(), True),
    StructField("vchr_status", StringType(), True),
    StructField("item_nbr", StringType(), True),
    StructField("item_desc", StringType(), True),
    StructField("thermo_item_nbr", StringType(), True),
    StructField("supplier_cd", StringType(), True),
    StructField("supplier_name", StringType(), True),
    StructField("supplier_type_cd", StringType(), True),
    StructField("buyer_cd", StringType(), True),
    StructField("document_desc", StringType(), True),
    StructField("invc_txn_type", StringType(), True),
    StructField("buyer_nm", StringType(), True),
    StructField("co_cd", StringType(), True),
    StructField("co_name", StringType(), True),
    StructField("hfm_entity", StringType(), True),
    StructField("business_unit", StringType(), True),
    StructField("lcr_flag", StringType(), True),
    StructField("lcr_region", StringType(), True),
    StructField("vomi_flag", StringType(), True),
    StructField("payment_compliance_flg", StringType(), True),
    StructField("po_curncy_cd", StringType(), True),
    StructField("co_curncy_cd", StringType(), True),
    StructField("post_yr_mth_nbr", StringType(), True),
    StructField("invc_entry_dt", StringType(), True),
    StructField("paymt_due_dt", StringType(), True),
    StructField("suplr_invc_dt", StringType(), True),
    StructField("aprval_dt", StringType(), True),
    StructField("txn_orig_id", StringType(), True),
    StructField("suplr_invc_nbr", StringType(), True),
    StructField("invc_apprv_id", StringType(), True),
    StructField("unit_prc", DoubleType(), True),
    StructField("invc_qty", DoubleType(), True),
    StructField("base_qty", DoubleType(), True),
    StructField("invc_txn_amt", DoubleType(), True),
    StructField("invc_co_amt", DoubleType(), True),
    StructField("invc_txn_pmar_amt", DoubleType(), True),
    StructField("invc_co_pmar_amt", DoubleType(), True),
    StructField("unit_prc_pmar_amt", DoubleType(), True),
    StructField("txn_curncy_mth_rt", DoubleType(), True),
    StructField("co_curncy_mth_rt", DoubleType(), True),
    StructField("uom_conv_factor", DoubleType(), True),
    StructField("invc_uom_cd", StringType(), True),
    StructField("base_uom_cd", StringType(), True),
    StructField("profit_cntr", StringType(), True),
    StructField("div_cd", StringType(), True),
    StructField("site_cd", StringType(), True),
    StructField("site_name", StringType(), True),
    StructField("reporting_site", StringType(), True),
    StructField("warehouse", StringType(), True),
    StructField("warehouse_nm", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("nature", StringType(), True),
    StructField("inv_flg", StringType(), True),
    StructField("inv_flg_text", StringType(), True),
    StructField("spend_type_cd", StringType(), True),
    StructField("po_paymt_terms_cd", StringType(), True),
    StructField("po_paymt_terms_desc", StringType(), True),
    StructField("suplr_paymt_terms_cd", StringType(), True),
    StructField("suplr_paymt_terms_desc", StringType(), True),
    StructField("fk_orig", StringType(), True),
    StructField("floor_stock_cd", StringType(), True),
    StructField("contract_flag", StringType(), True),
    StructField("contract_type", StringType(), True),
    StructField("contract_start_date", StringType(), True),
    StructField("contract_end_date", StringType(), True),
    StructField("erp_commondity_cd", StringType(), True),
    StructField("erp_commondity_nm", StringType(), True),
    StructField("sec_supp_cd", StringType(), True),
    StructField("part_rev_no", StringType(), True),
    StructField("cost_centre_cd", StringType(), True),
    StructField("cost_centre_nm", StringType(), True),
    StructField("vendor_mat_no", StringType(), True),
    StructField("gl_acct_id", StringType(), True),
    StructField("gl_acct_nm", StringType(), True),
    StructField("pass_through_field", StringType(), True),
    StructField("pass_through_line", StringType(), True),
    StructField("inv_line_desc", StringType(), True),
    StructField("remit_to_addr_line_1", StringType(), True),
    StructField("remit_to_addr_line_2", StringType(), True),
    StructField("remit_to_addr_line_3", StringType(), True),
    StructField("remit_to_addr_line_4", StringType(), True),
    StructField("remit_to_city_nm", StringType(), True),
    StructField("remit_to_st_cd", StringType(), True),
    StructField("remit_to_rgn_cd", StringType(), True),
    StructField("remit_to_rgn_nm", StringType(), True),
    StructField("remit_to_cntry_cd", StringType(), True),
    StructField("remit_to_cntry_nm", StringType(), True),
    StructField("suplr_nm_src", StringType(), True),
    StructField("rpt_flex1", StringType(), True),
    StructField("invc_txn_amt_clsfctn", StringType(), True),
    StructField("supplier_segment", StringType(), True),
    StructField("ap_payment_term_cd", StringType(), True),
    StructField("ap_payment_term_desc", StringType(), True),
    StructField("actual_payment_dt", StringType(), True),
    StructField("source_country", StringType(), True)
])

def get_dw_ap_sla_aging_invoice_ca_vw(raw_unity_catalog: str):
    """
    Returns latest snapshot per invoice_id from dw_ap_sla_aging_invoice_ca.
    Args:
        raw_unity_catalog (str): Unity Catalog schema for raw tables.
    Returns:
        DataFrame: Latest snapshot per invoice_id.
    """
    df = spark.read.table(f"{raw_unity_catalog}.dw_ap_sla_aging_invoice_ca")
    window_spec = Window.partitionBy("invoice_id").orderBy(F.col("snapshot_captured_date").desc())
    df = df.withColumn("RowNum", F.row_number().over(window_spec))
    df = df.filter(F.col("RowNum") == 1)
    return df

dw_ap_sla_aging_invoice_ca_vw = get_dw_ap_sla_aging_invoice_ca_vw(raw_unity_catalog)

def get_dasedc(raw_unity_catalog: str):
    """
    Returns latest expense dist per distribution from dw_ap_sla_expense_dist_cf.
    Args:
        raw_unity_catalog (str): Unity Catalog schema for raw tables.
    Returns:
        DataFrame: Latest expense dist per distribution.
    """
    df = spark.read.table(f"{raw_unity_catalog}.dw_ap_sla_expense_dist_cf")
    window_spec = Window.partitionBy(
        "invoice_distribution_id",
        "gl_balancing_segment",
        "cost_center_segment",
        "gl_segment1",
        "invoice_id",
        "distribution_line_number",
        "invoice_line_number",
        "invoice_accounting_date",
        "transaction_amount"
    ).orderBy(F.col("xla_manual_override_flag").desc())
    df = df.withColumn("RowNum", F.row_number().over(window_spec))
    df = df.filter(F.col("RowNum") == 1)
    return df

dasedc = get_dasedc(raw_unity_catalog)

# -- Read all required lookup and dimension tables
dpd = spark.read.table(f"{raw_unity_catalog}.dw_party_d")
dssd = spark.read.table(f"{raw_unity_catalog}.dw_supplier_site_d")
diodt = spark.read.table(f"{raw_unity_catalog}.dw_internal_org_d_tl")
datdt = spark.read.table(f"{raw_unity_catalog}.dw_ap_terms_d_tl")
dnad = spark.read.table(f"{raw_unity_catalog}.dw_natural_account_d")
daspc = spark.read.table(f"{raw_unity_catalog}.dw_ap_sla_payments_cf")
dgsdt_cc = spark.read.table(f"{raw_unity_catalog}.dw_gl_segment_d_tl")
dgsdt_cc_sla = spark.read.table(f"{raw_unity_catalog}.dw_gl_segment_d_tl")
dgsdt_cd = spark.read.table(f"{raw_unity_catalog}.dw_gl_segment_d_tl")
dgsdt_cd_sla = spark.read.table(f"{raw_unity_catalog}.dw_gl_segment_d_tl")
dgccd = spark.read.table(f"{raw_unity_catalog}.dw_gl_code_combination_d")
dgccd_sla = spark.read.table(f"{raw_unity_catalog}.dw_gl_code_combination_d")
dgsdt_acct = spark.read.table(f"{raw_unity_catalog}.dw_gl_segment_d_tl")
dgsdt_acct_sla = spark.read.table(f"{raw_unity_catalog}.dw_gl_segment_d_tl")
dgsdt_subacct = spark.read.table(f"{raw_unity_catalog}.dw_gl_segment_d_tl")
dgsdt_subacct_sla = spark.read.table(f"{raw_unity_catalog}.dw_gl_segment_d_tl")
edp_lkup = spark.read.table(f"{edp_lkp_unity_catalog}.edp_lkup")
edp_lkup_div = spark.read.table(f"{edp_lkp_unity_catalog}.edp_lkup")
edp_lkup_div_1 = spark.read.table(f"{edp_lkp_unity_catalog}.edp_lkup")
edp_lkup_payment = spark.read.table(f"{edp_lkp_unity_catalog}.edp_lkup")
comp = spark.read.table(f"{dims_unity_catalog}.dim_wf_company")

# -- Prepare daspc (payments) for join
daspc_filtered = daspc.filter(
    F.col("check_void_date") == '1901-01-01T00:00:00.000+00:00'
).groupBy("invoice_id", "invoice_distribution_id").agg(
    F.first("check_date").alias("check_date")
)

# -- Join all tables as per SQL logic
fa_df = dw_ap_sla_aging_invoice_ca_vw.alias("dasaic") \
    .join(
        dasedc.alias("dasedc"),
        ["invoice_id"], "left"
    ) \
    .join(
        dpd.alias("dpd"),
        F.col("dasaic.supplier_party_id") == F.col("dpd.party_id"),
        "left"
    ) \
    .join(
        dssd.alias("dssd"),
        ["supplier_site_id"], "left"
    ) \
    .join(
        diodt.alias("diodt"),
        F.col("dasaic.payables_bu_id") == F.col("diodt.organization_id"),
        "left"
    ) \
    .join(
        datdt.alias("datdt"),
        ["PAYMENT_TERMS_ID"], "left"
    ) \
    .join(
        dnad.alias("dnad"),
        ["natural_account_segment"], "left"
    ) \
    .join(
        daspc_filtered.alias("daspc"),
        ["invoice_distribution_id", "invoice_id"], "left"
    ) \
    .join(
        dgsdt_cc.alias("dgsdt_cc"),
        (F.col("dasedc.cost_center_segment") == F.col("dgsdt_cc.gl_segment_code")) &
        (F.col("dgsdt_cc.gl_segment_valueset_code") == F.lit('Center FS_CCG_COA')),
        "left"
    ) \
    .join(
        dgsdt_cc_sla.alias("dgsdt_cc_sla"),
        (F.col("dasaic.cost_center_segment") == F.col("dgsdt_cc_sla.gl_segment_code")) &
        (F.col("dgsdt_cc_sla.gl_segment_valueset_code") == F.lit('Center FS_CCG_COA')),
        "left"
    ) \
    .join(
        dgsdt_cd.alias("dgsdt_cd"),
        (F.col("dasedc.gl_balancing_segment") == F.col("dgsdt_cd.gl_segment_code")) &
        (F.col("dgsdt_cd.gl_segment_valueset_code") == F.lit('Bal Entity FS_CCG_COA')),
        "left"
    ) \
    .join(
        dgsdt_cd_sla.alias("dgsdt_cd_sla"),
        (F.col("dasaic.gl_balancing_segment") == F.col("dgsdt_cd_sla.gl_segment_code")) &
        (F.col("dgsdt_cd_sla.gl_segment_valueset_code") == F.lit('Bal Entity FS_CCG_COA')),
        "left"
    ) \
    .join(
        dgccd.alias("dgccd"),
        F.col("dasedc.gl_code_combination_id") == F.col("dgccd.code_combination_id"),
        "left"
    ) \
    .join(
        dgccd_sla.alias("dgccd_sla"),
        F.col("dasaic.gl_code_combination_id") == F.col("dgccd_sla.code_combination_id"),
        "left"
    ) \
    .join(
        dgsdt_acct.alias("dgsdt_acct"),
        (F.col("dasedc.natural_account_segment") == F.col("dgsdt_acct.gl_segment_code")) &
        (F.col("dgsdt_acct.gl_segment_valueset_code") == F.lit('Account FS_CCG_COA')),
        "left"
    ) \
    .join(
        dgsdt_acct_sla.alias("dgsdt_acct_sla"),
        (F.col("dasaic.natural_account_segment") == F.col("dgsdt_acct_sla.gl_segment_code")) &
        (F.col("dgsdt_acct_sla.gl_segment_valueset_code") == F.lit('Account FS_CCG_COA')),
        "left"
    ) \
    .join(
        dgsdt_subacct.alias("dgsdt_subacct"),
        (F.col("dasedc.gl_segment1") == F.col("dgsdt_subacct.gl_segment_code")) &
        (F.col("dgsdt_subacct.gl_segment_valueset_code") == F.lit('Sub Account FS_CCG_COA')),
        "left"
    ) \
    .join(
        dgsdt_subacct_sla.alias("dgsdt_subacct_sla"),
        (F.col("dasaic.gl_segment1") == F.col("dgsdt_subacct_sla.gl_segment_code")) &
        (F.col("dgsdt_subacct_sla.gl_segment_valueset_code") == F.lit('Sub Account FS_CCG_COA')),
        "left"
    ) \
    .join(
        edp_lkup.alias("edp_lkup"),
        (F.col("edp_lkup.lkup_key_02") == F.coalesce(F.col("dasedc.cost_center_segment"), F.col("dasaic.cost_center_segment"))) &
        (F.col("edp_lkup.lkup_typ_nm") == F.lit('TFS_CC_TO_DIV')) &
        (F.lower(F.col("edp_lkup.lkup_key_01")) == F.lit('usorafin')),
        "left"
    ) \
    .join(
        edp_lkup_div.alias("edp_lkup_div"),
        (F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")) == F.col("edp_lkup_div.lkup_key_02")) &
        (F.coalesce(F.col("dasedc.cost_center_segment"), F.col("dasaic.cost_center_segment")) == F.col("edp_lkup_div.lkup_key_03")) &
        (F.concat_ws('-', F.coalesce(F.col("dasedc.natural_account_segment"), F.col("dasaic.natural_account_segment")), F.coalesce(F.col("dasedc.gl_segment1"), F.col("dasaic.gl_segment1"))) == F.col("edp_lkup_div.lkup_key_04")) &
        (F.col("edp_lkup_div.lkup_typ_nm") == F.lit('TFS_COCD_CCNTR_GLACCT_TO_DIVCD')) &
        (F.upper(F.col("edp_lkup_div.lkup_key_01")) == F.lit('USORAFIN')),
        "left"
    ) \
    .join(
        edp_lkup_div_1.alias("edp_lkup_div_1"),
        (F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")) == F.col("edp_lkup_div_1.lkup_key_02")) &
        (F.coalesce(F.col("dasedc.cost_center_segment"), F.col("dasaic.cost_center_segment")) == F.col("edp_lkup_div_1.lkup_key_03")) &
        (F.col("edp_lkup_div_1.lkup_key_04").isNull()) &
        (F.col("edp_lkup_div_1.lkup_typ_nm") == F.lit('TFS_COCD_CCNTR_GLACCT_TO_DIVCD')) &
        (F.upper(F.col("edp_lkup_div_1.lkup_key_01")) == F.lit('USORAFIN')),
        "left"
    ) \
    .join(
        edp_lkup_payment.alias("edp_lkup_payment"),
        (F.upper(F.col("edp_lkup_payment.lkup_key_02")) == F.upper(F.col("datdt.payment_term_name"))) &
        (F.col("edp_lkup_payment.lkup_typ_nm") == F.lit('PAYMENT_TERMS_MAPPING')) &
        (F.upper(F.col("edp_lkup_payment.lkup_key_01")) == F.lit('USORAFIN')),
        "left"
    ) \
    .join(
        comp.alias("comp"),
        F.col("comp.co_cd") == F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")),
        "left"
    )

# -- Filtering logic as per SQL WHERE clause
fa_df = fa_df.filter(
    (F.year(F.col("dasaic.invoice_accounting_date")) >= F.year(F.current_timestamp()) - 3) &
    (F.col("dasaic.invoice_source_code") != 'Receivables') &
    (
        ~(
            (F.col("dasedc.invoice_source_code") == 'MAINFRAME') &
            (
                F.col("dasedc.invoice_description").rlike("^DS.*") |
                F.col("dasedc.invoice_description").rlike("^DR.*") |
                F.col("dasedc.invoice_description").rlike("^PE.*") |
                F.col("dasedc.invoice_description").rlike("^PR.*") |
                F.col("dasedc.invoice_description").rlike("^PS.*") |
                F.col("dasedc.invoice_description").rlike("^RG.*") |
                F.col("dasedc.invoice_description").rlike("^TR.*")
            )
        ) | F.col("dasedc.invoice_description").isNull()
    ) &
    (
        (
            (F.col("dasedc.natural_account_segment").between(4000, 8999)) |
            (F.col("dasedc.natural_account_segment").isNull())
        ) |
        (
            (F.col("dasedc.cost_center_segment").between(9000, 9999)) |
            (F.col("dasedc.cost_center_segment").isNull())
        )
    ) &
    (~F.regexp_replace(F.col("dgccd.concat_segments"), '\\.', '').isin(
        '4009999110011000000000000000','4009999111011104000000000000','4009999111011105000000000000',
        '4009999111011106000000000000','4009999129012946000000000000','4009999138013800000000000000',
        '4009999172017211000000000000','4009999200020004000000000000','4009999240024042000000000000',
        '4009999240024047000000000000','7009801138013800000000000000','7009801210021031000000000000',
        '7009801210021079000000000000'
    )) &
    (~F.regexp_replace(F.col("dgccd_sla.concat_segments"), '\\.', '').isin(
        '4009999110011000000000000000','4009999111011104000000000000','4009999111011105000000000000',
        '4009999111011106000000000000','4009999129012946000000000000','4009999138013800000000000000',
        '4009999172017211000000000000','4009999200020004000000000000','4009999240024042000000000000',
        '4009999240024047000000000000','7009801138013800000000000000','7009801210021031000000000000',
        '7009801210021079000000000000'
    )) &
    (~F.col("dpd.supplier_number").isin('00032238','00032239','00080463','90000147'))
)

# -- Select and transform columns as per SQL SELECT
fa_df = fa_df.select(
    F.lit(None).cast(StringType()).alias("document_type"),
    F.lit(None).cast(StringType()).alias("txn_ref_nbr"),
    F.date_format(F.col("dasaic.invoiced_on_date"), 'yyyyMM').alias("invc_entry_period"),
    F.lit(None).cast(StringType()).alias("po_nbr"),
    F.lit(None).cast(StringType()).alias("po_line_nbr"),
    F.lit("usorafin").alias("src_sys_cd"),
    F.col("dasaic.invoice_id").cast(StringType()).alias("vchr_nbr"),
    F.concat_ws("-", F.col("dasedc.invoice_line_number"), F.col("dasedc.distribution_line_number")).cast(StringType()).alias("vchr_line_nbr"),
    F.date_format(F.col("dasedc.invoice_accounting_date"), 'yyyy').alias("fscl_yr_nbr"),
    F.col("dasaic.invoice_type_code").alias("vchr_type_cd"),
    F.lit(None).cast(StringType()).alias("vchr_status"),
    F.lit(None).cast(StringType()).alias("item_nbr"),
    F.lit(None).cast(StringType()).alias("item_desc"),
    F.lit(None).cast(StringType()).alias("thermo_item_nbr"),
    F.concat_ws("_", F.coalesce(F.col("dpd.supplier_number"), F.lit(0)), F.col("dssd.supplier_site_id")).alias("supplier_cd"),
    F.col("dpd.party_name").alias("supplier_name"),
    F.lit(None).cast(StringType()).alias("supplier_type_cd"),
    F.lit(None).cast(StringType()).alias("buyer_cd"),
    F.lit(None).cast(StringType()).alias("document_desc"),
    F.lit(None).cast(StringType()).alias("invc_txn_type"),
    F.lit(None).cast(StringType()).alias("buyer_nm"),
    F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")).alias("co_cd"),
    F.col("comp.co_nm").alias("co_name"),  # FIXED: Use co_nm as co_name
    F.col("edp_lkup.lkup_val_03").alias("hfm_entity"),
    F.col("diodt.organization_name").alias("business_unit"),
    F.lit(None).cast(StringType()).alias("lcr_flag"),
    F.lit(None).cast(StringType()).alias("lcr_region"),
    F.lit(None).cast(StringType()).alias("vomi_flag"),
    F.lit(None).cast(StringType()).alias("payment_compliance_flg"),
    F.col("dasaic.transaction_currency_code").alias("po_curncy_cd"),
    F.col("dasaic.ledger_currency_code").alias("co_curncy_cd"),
    F.date_format(F.col("dasedc.invoice_accounting_date"), 'yyyyMM').alias("post_yr_mth_nbr"),
    F.date_format(F.col("dasaic.invoiced_on_date"), 'yyyyMMdd').alias("invc_entry_dt"),
    F.date_format(F.col("dasaic.invoice_schedule_due_date"), 'yyyyMMdd').alias("paymt_due_dt"),
    F.date_format(F.col("dasaic.invoice_accounting_date"), 'yyyyMMdd').alias("suplr_invc_dt"),
    F.lit(None).cast(StringType()).alias("aprval_dt"),
    F.lit(None).cast(StringType()).alias("txn_orig_id"),
    F.col("dasaic.invoice_number").alias("suplr_invc_nbr"),
    F.lit(None).cast(StringType()).alias("invc_apprv_id"),
    F.col("dasedc.transaction_amount").cast(DoubleType()).alias("unit_prc"),
    F.lit(1).cast(DoubleType()).alias("invc_qty"),
    F.lit(None).cast(DoubleType()).alias("base_qty"),
    F.col("dasedc.transaction_amount").cast(DoubleType()).alias("invc_txn_amt"),
    F.col("dasedc.transaction_amount").cast(DoubleType()).alias("invc_co_amt"),
    F.lit(None).cast(DoubleType()).alias("invc_txn_pmar_amt"),
    F.lit(0).cast(DoubleType()).alias("invc_co_pmar_amt"),
    F.lit(0).cast(DoubleType()).alias("unit_prc_pmar_amt"),
    F.lit(0).cast(DoubleType()).alias("txn_curncy_mth_rt"),
    F.lit(0).cast(DoubleType()).alias("co_curncy_mth_rt"),
    F.lit(None).cast(DoubleType()).alias("uom_conv_factor"),
    F.lit(None).cast(StringType()).alias("invc_uom_cd"),
    F.lit(None).cast(StringType()).alias("base_uom_cd"),
    F.lit(None).cast(StringType()).alias("profit_cntr"),
    F.when(
        F.coalesce(F.col("edp_lkup_div.lkup_val_01"), F.col("edp_lkup_div_1.lkup_val_01")).isNull(),
        F.when(F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")) == 400, "CCG Group")
         .when(F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")) == 700, "Corporate")
    ).otherwise(F.coalesce(F.col("edp_lkup_div.lkup_val_01"), F.col("edp_lkup_div_1.lkup_val_01"))).alias("div_cd"),
    F.lit(None).cast(StringType()).alias("site_cd"),
    F.lit(None).cast(StringType()).alias("site_name"),
    F.lit(None).cast(StringType()).alias("reporting_site"),
    F.lit(None).cast(StringType()).alias("warehouse"),
    F.lit(None).cast(StringType()).alias("warehouse_nm"),
    F.lit(None).cast(StringType()).alias("unit"),
    F.lit(None).cast(StringType()).alias("nature"),
    F.lit(None).cast(StringType()).alias("inv_flg"),
    F.lit(None).cast(StringType()).alias("inv_flg_text"),
    F.lit("Indirect").alias("spend_type_cd"),
    F.lit(None).cast(StringType()).alias("po_paymt_terms_cd"),
    F.lit(None).cast(StringType()).alias("po_paymt_terms_desc"),
    F.col("datdt.payment_term_name").alias("suplr_paymt_terms_cd"),
    F.coalesce(F.col("datdt.payment_term_description"), F.col("edp_lkup_payment.lkup_val_01")).alias("suplr_paymt_terms_desc"),
    F.lit(None).cast(StringType()).alias("fk_orig"),
    F.lit(None).cast(StringType()).alias("floor_stock_cd"),
    F.lit(None).cast(StringType()).alias("contract_flag"),
    F.lit(None).cast(StringType()).alias("contract_type"),
    F.lit(None).cast(StringType()).alias("contract_start_date"),
    F.lit(None).cast(StringType()).alias("contract_end_date"),
    F.lit(None).cast(StringType()).alias("erp_commondity_cd"),
    F.lit(None).cast(StringType()).alias("erp_commondity_nm"),
    F.lit(None).cast(StringType()).alias("sec_supp_cd"),
    F.lit(None).cast(StringType()).alias("part_rev_no"),
    F.coalesce(F.col("dasedc.cost_center_segment"), F.col("dasaic.cost_center_segment")).alias("cost_centre_cd"),
    F.coalesce(F.col("dgsdt_cc.gl_segment_description"), F.col("dgsdt_cc_sla.gl_segment_description")).alias("cost_centre_nm"),
    F.lit(None).cast(StringType()).alias("vendor_mat_no"),
    F.concat_ws("-", F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")),
                     F.coalesce(F.col("dasedc.natural_account_segment"), F.col("dasaic.natural_account_segment")),
                     F.coalesce(F.col("dasedc.gl_segment1"), F.col("dasaic.gl_segment1"))).alias("gl_acct_id"),
    F.concat_ws("-", F.coalesce(F.col("dgsdt_acct.gl_segment_description"), F.col("dgsdt_acct_sla.gl_segment_description")),
                     F.coalesce(F.col("dgsdt_subacct.gl_segment_description"), F.col("dgsdt_subacct_sla.gl_segment_description"))).alias("gl_acct_nm"),
    F.lit(None).cast(StringType()).alias("pass_through_field"),
    F.lit(None).cast(StringType()).alias("pass_through_line"),
    F.col("dasedc.invoice_description").cast(StringType()).alias("inv_line_desc"),
    F.col("dssd.address1").alias("remit_to_addr_line_1"),
    F.col("dssd.address2").alias("remit_to_addr_line_2"),
    F.lit(None).cast(StringType()).alias("remit_to_addr_line_3"),
    F.lit(None).cast(StringType()).alias("remit_to_addr_line_4"),
    F.col("dssd.city").alias("remit_to_city_nm"),
    F.col("dssd.state").alias("remit_to_st_cd"),
    F.lit(None).cast(StringType()).alias("remit_to_rgn_cd"),
    F.lit(None).cast(StringType()).alias("remit_to_rgn_nm"),
    F.col("dssd.country").alias("remit_to_cntry_cd"),
    F.lit(None).cast(StringType()).alias("remit_to_cntry_nm"),
    F.lit(None).cast(StringType()).alias("suplr_nm_src"),
    F.lit(None).cast(StringType()).alias("rpt_flex1"),
    F.lit(None).cast(StringType()).alias("invc_txn_amt_clsfctn"),
    F.lit(None).cast(StringType()).alias("supplier_segment"),
    F.col("datdt.payment_term_name").alias("ap_payment_term_cd"),
    F.coalesce(F.col("datdt.payment_term_description"), F.col("edp_lkup_payment.lkup_val_01")).alias("ap_payment_term_desc"),
    F.date_format(F.col("daspc.check_date"), 'yyyy-MM-dd').alias("actual_payment_dt"),
    F.lit("NA").alias("source_country")
)

# -- Data Quality Checks: Remove negative transaction amounts and enforce allowed payment terms
allowed_terms = ["Net 30", "Net 60", "Immediate", "Net 0", "Net 999"]
fa_df = fa_df.filter(
    (F.col("invc_txn_amt") >= 0) &
    (F.col("ap_payment_term_cd").isin(allowed_terms))
)

# -- Enforce schema consistency before write
fa_df = enforce_schema(fa_df, expected_schema)

# -- Write to target table path in specified format, partitioned by partition_key
write_mode = "overwrite"
write_options = {"compression": compression}
if table_format == "delta":
    fa_df.write.format("delta").mode(write_mode).option("overwriteSchema", "true").partitionBy(partition_key).options(**write_options).save(target_table_path)
elif table_format == "parquet":
    fa_df.write.format("parquet").mode(write_mode).partitionBy(partition_key).options(**write_options).save(target_table_path)
else:
    raise Exception(f"Value {table_format} not allowed for table_format")

# -- End of script
# spark.stop()  # Do not stop SparkSession in Databricks